In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
import imageio.v2 as imageio
import glob
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm, PowerNorm
import datetime
from matplotlib.ticker import MaxNLocator

# Data-loading/combining/sorting helpers, moved out to their own module (as
# in the PR notebook) -- see Data_Opening_Tools_module.py.
import Data_Opening_Tools_module as DT




In [ ]:
# Definitions

def tanh(x):
    """Hyperbolic tangent (equivalent to np.tanh). Used in a couple of
    activity-plotting cells further down."""
    return (np.exp(x) - np.exp(-x)) / (np.exp(x) + np.exp(-x))


def mu_sigma_per_HSM(used_seeds, Data_sorted):
    """
    Compute the per-HSM mean and standard deviation of Data_sorted,
    assuming Data_sorted is laid out as repeating blocks of
    len(used_seeds) entries per HSM value.

    NOTE: currently unused anywhere in this notebook. FIX applied: the loop
    previously used an undefined name `seeds` where the parameter is
    `used_seeds` (NameError), and a stray bare `L` statement (leftover
    typo) has been removed. Left as-is: `one_set_of_HSMs` reads a global
    `HSMs_sorted`, which nothing in this notebook actually assigns at
    module level (the data-opening functions return a *_sorted tuple
    locally, but nothing binds it to a bare `HSMs_sorted` global) -- so
    calling this function would still raise NameError until that's wired
    up. Worth deciding whether HSMs should be passed in explicitly instead,
    if you start using this function.

    Args:
        used_seeds: the seeds whose block length blocks up Data_sorted
        Data_sorted: values sorted such that each HSM's block of
                     len(used_seeds) entries is contiguous

    Returns:
        (Data_mus, Data_sigmas, one_set_of_HSMs)
    """
    Data_mus = []
    Data_sigmas = []

    for i in range(len(used_seeds)):
        Data_perseed = Data_sorted[i::len(used_seeds)]

        mu = np.mean(Data_perseed)
        sigma = np.std(Data_perseed)

        Data_mus.append(mu)
        Data_sigmas.append(sigma)

    one_set_of_HSMs = HSMs_sorted[0:11]

    return Data_mus, Data_sigmas, one_set_of_HSMs


def distance_plots(running_time, save_dir, distances_sorted, seeds_sorted, HSMs_sorted):
    """
    Plot and save log-distance-vs-time for each (seed, HSM) pair.
    Currently unused anywhere in this notebook, kept for reference.
    """
    for index, (seed, HSM) in enumerate(zip(seeds_sorted, HSMs_sorted)):
        plt.figure(figsize=(10, 5))
        plt.semilogy(running_time, distances_sorted[index, :])
        plt.xlabel("Time")
        plt.ylabel("log(distances between X2 and X1)")
        plt.title(f"{index}. Lyapunov divergence for heterogeneity HSM {HSM}, seed {seed}")
        plt.grid(True)

        filename = f"plot_{index}_HSM{HSM:.2f}_seed_{seed:.2f}.png"
        filepath = os.path.join(save_dir, filename)
        plt.savefig(filepath)
        plt.close()  # close figure to save memory

    plt.show()


# Data Opening/Sorting

In [ ]:
# Step 1: Find all result files
# files = sorted(glob.glob("C:/Users/Lenovo/Documents/B. Master Neurophysics/Stage/Presentations and reports and code/code/LLE/LLE data/All_LLEs_MSHs_Gains_3000N_W_0-2_I_0-1/Different_gains_LLEs_0-2_W_and_0-1_Init_3000N_rank*.npz"))
# files = sorted(glob.glob("C:/Users/Lenovo/Documents/B. Master Neurophysics/Stage/Presentations and reports and code/code/LLE/LLE data/ALL_Results_Different_gains_LLEs_0-9seeds_W_and_Init/Different_gains_LLEs_0-9seeds_W_and_Init_rank*.npz"))

files = sorted(glob.glob("C:/Users/Lenovo/Documents/B. Master Neurophysics/Stage/Presentations and reports and code/code/LLE/LLE data/Zoomed in LLEs gain 1-3 HSM 0-0.1/LLEs_HSMs0-0.1_Gains1-3_W5-9_and_Init0-3/Zoomed_in_gains1-3_HSMs0-0.1_LLEs_W_and_Init_rank*.npz"))

LLEs_sorted, Gains_sorted, Weights_Seeds_sorted, Initialisation_Seeds_sorted, HSMs_sorted = DT.gains_combined_sorted_LLEs(files)


# Data Investigation

In [ ]:
print(f"{Weights_Seeds_sorted[0:500]}, length of ordered seed 0 = 270 (10 (init_seeds) times 27 (HSM)), shape of Weight_Seeds_sorted: {Weights_Seeds_sorted.shape}, should be equal to n_weights_seeds = range(6,8) (shape of 2) X n_init_seeds = range(10) (shape of 10) X HSM(HSMs) = np.arange(0, 5.2, 0.2) + value 10 (shape of 26 + 1= 27) X gains = np.aragne(0, 10.2, 0.2) (shape of 51) = 27540")
print(f"{Initialisation_Seeds_sorted[0:54]}, length of ordered seeds 0 = 27, shape of Weight_Seeds_sorted: {Initialisation_Seeds_sorted.shape}")
print(f" {HSMs_sorted}, length of one times all values = 27, shape of HSMs_sorted: {HSMs_sorted.shape}")
print(f"{Gains_sorted[1000:1621]}, length of 0 values = 1620 because of 27 HSMs X 10 init_seeds X 6 Weights_seeds, shape of Gains_sorted: {Gains_sorted.shape}")
print(f"{LLEs_sorted[0:54]}, shape of LLEs_sorted: {LLEs_sorted.shape}")

# LLE Heatmap

In [19]:
#multiplying MLE values with 1000 to get 1/s^-1 unit
LLEs_sorted *= 1000

In [ ]:
# # --- Set GLOBAL min/max for BOTH datasets (symmetric around 0) ---
# GLOBAL_VMAX = 125  # Set this to the maximum absolute MLE value across BOTH datasets
# GLOBAL_VMIN = -GLOBAL_VMAX  # Symmetric around 0

plt.rcParams['axes.titlesize'] = 30
plt.rcParams['axes.labelsize'] = 36
plt.rcParams['xtick.labelsize'] = 30
plt.rcParams['ytick.labelsize'] = 30
plt.rcParams['legend.fontsize'] = 36
plt.rcParams['figure.titlesize'] = 30
plt.rcParams['legend.title_fontsize'] = 30

#tick size:
plt.rcParams['xtick.major.size'] = 8    # length of major ticks
plt.rcParams['ytick.major.size'] = 8
plt.rcParams['xtick.major.width'] = 2  # width of major ticks
plt.rcParams['ytick.major.width'] = 2



# Create custom colormap: Blue -> White -> Red
dark_red = "#8B0000"
white = "white"
dark_blue = "#013220"

colors = [dark_blue, white, dark_red]
n_bins = 251
cmap_name = 'custom_seismic'
cm = LinearSegmentedColormap.from_list(cmap_name, colors, N=n_bins)




mask = (HSMs_sorted != 10) 

df = pd.DataFrame({
    'G': Gains_sorted[mask],
    'W': Weights_Seeds_sorted[mask],
    'I': Initialisation_Seeds_sorted[mask],
    'HSM': HSMs_sorted[mask],
    'L': LLEs_sorted[mask]
})

df_mean = df.groupby(['G', 'W', 'I', 'HSM'], as_index=False)['L'].mean()
heatmap_data = df_mean.pivot_table(index='G', columns='HSM', values='L', aggfunc='mean')

# Set the (G=0, HSM=0) cell to NaN → will render as white
if 0 in heatmap_data.index and 0 in heatmap_data.columns:
    heatmap_data.loc[0, 0] = np.nan

Universe_VMAX = 125
Universe_VMIN = -100

# for if you do not want to compare between 1000N and 3000N
GLOBAL_VMIN = df_mean['L'].min()
GLOBAL_VMAX = df_mean['L'].max()


# Create a norm that centers at 0 with GLOBAL limits
norm = TwoSlopeNorm(vmin=Universe_VMIN, 
                    vcenter=0, 
                    vmax=Universe_VMAX)

plt.figure(figsize=(14,10))

cmap = plt.get_cmap('seismic').copy()
cmap.set_bad(color='white')

mesh = plt.pcolormesh(
    heatmap_data.columns, 
    heatmap_data.index, 
    heatmap_data.values,
    shading='auto', 
    cmap=cmap,  # Use custom colormap
    norm=norm  # Use centered normalization
)

cbar = plt.colorbar(mesh, label='Maximal Lyapunov Exponent (MLE) ($s^{-1}$)')
cbar.ax.yaxis.set_label_coords(5.0, 0.5)  # Adjust position if needed
cbar.set_label('Maximal Lyapunov Exponent (MLE) ($s^{-1}$)', rotation=270, labelpad=20)


# Crop colorbar display to local data range (norm stays global)
local_min = df_mean['L'].min()
local_max = df_mean['L'].max()

# Convert local data values to norm space (0-1) for the colorbar axis
norm_min = norm(local_min)
norm_max = norm(local_max)

cbar.ax.set_ylim(norm_min, norm_max)

# Ticks within local range only
n_ticks_per_side = 5
neg_ticks = np.linspace(local_min, 0, 2 + 1)
pos_ticks = np.linspace(0, local_max, n_ticks_per_side + 1)
tick_values = np.unique(np.concatenate([neg_ticks, pos_ticks]))

cbar.set_ticks(tick_values)
cbar.set_ticklabels([f'{v:.0f}' for v in tick_values])


plt.xlabel('Heterogeneity of Synaptic Mean (HSM)', labelpad=20)
plt.ylabel('Gain (g)')
# plt.title('Stability/Chaos for 1000N, 10 W seeds x 10 I seeds')

#plt.title('Stability/Chaos for 3000N, 3 Wseeds x 2 Iseeds')

timestamp = datetime.datetime.now().strftime("%Y-%m-%d_ %H;%M;%S")
filename = f"MLE_Map_gain_HSM_{timestamp}.png"
filepath = os.path.join(r"C:\Users\Lenovo\Documents\B. Master Neurophysics\Stage\Presentations and reports and code\code\LLE\MLE map PNGs for Gain and HSM", filename)

#filepath = os.path.join(r"C:\Users\Lenovo\Documents\B. Master Neurophysics\Stage\Presentations and reports and code\code\LLE\MLE map 3000N", filename)

ax = plt.gca()
ax.tick_params(axis='x', rotation=300)

#for increasing the width of the plot edge line
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(2)

ax.xaxis.set_major_locator(MaxNLocator(nbins=11))
ax.yaxis.set_major_locator(MaxNLocator(nbins=11))

#for increasing the width of the edge line of the colour bar
cbar.outline.set_linewidth(2)

plt.savefig(filepath, dpi=300)
plt.show()

In [ ]:
#Std heatmap

plt.rcParams['axes.titlesize'] = 30
plt.rcParams['axes.labelsize'] = 36
plt.rcParams['xtick.labelsize'] = 30
plt.rcParams['ytick.labelsize'] = 30
plt.rcParams['legend.fontsize'] = 36
plt.rcParams['figure.titlesize'] = 30
plt.rcParams['legend.title_fontsize'] = 30

#tick size:
plt.rcParams['xtick.major.size'] = 8    # length of major ticks
plt.rcParams['ytick.major.size'] = 8
plt.rcParams['xtick.major.width'] = 2  # width of major ticks
plt.rcParams['ytick.major.width'] = 2


mask = (HSMs_sorted != 10) 

# Put them into a DataFrame
df = pd.DataFrame({
    'G': Gains_sorted[mask],
    'W': Weights_Seeds_sorted[mask],
    'I': Initialisation_Seeds_sorted[mask],
    'HSM': HSMs_sorted[mask],
    'L': LLEs_sorted[mask]
})



# Average over W and I for each (G, HSM) combination
# (If you want to ignore W and I completely, comment the line below and uncomment the next one)
#df_mean = df.groupby(['G', 'HSM'], as_index=False)['L'].mean()
df_mean = df.groupby(['G', 'W', 'I', 'HSM'], as_index=False)['L'].mean()
df_std = df_mean.groupby(['G', 'HSM'], as_index=False)['L'].std()


heatmap_data = df_std.pivot_table(index='G', columns='HSM', values='L', aggfunc='mean')


# Set the (G=0, HSM=0) cell to NaN → will render as white
if 0 in heatmap_data.index and 0 in heatmap_data.columns:
    heatmap_data.loc[0, 0] = np.nan


# for if you do not want to compare between 1000N and 3000N
GLOBAL_VMIN = df_std['L'].min()
GLOBAL_VMAX = df_std['L'].max()

print(GLOBAL_VMAX, GLOBAL_VMIN)

plt.figure(figsize=(14,10))

# Use GLOBAL vmin/vmax instead of data-specific min/max
norm = PowerNorm(
    gamma=0.25,
    vmin=GLOBAL_VMIN,  # Changed
    vmax=GLOBAL_VMAX   # Changed
)

cmap = plt.get_cmap('cividis').copy()
cmap.set_bad(color='white')

mesh = plt.pcolormesh(
    heatmap_data.columns,
    heatmap_data.index,
    heatmap_data.values,
    shading='auto',
    cmap=cmap,
    norm=norm
)

cbar = plt.colorbar(mesh)
cbar.set_label('standard deviation ($\sigma$) of MLE', rotation=270, labelpad=20)
cbar.ax.yaxis.set_label_coords(5.0, 0.5)  # Adjust position if needed


cbar.ax.minorticks_off()

n_ticks = 6  
tick_positions = np.linspace(0, 1, n_ticks)

# Use GLOBAL values for tick calculation
data_min = GLOBAL_VMIN  # Changed
data_max = GLOBAL_VMAX  # Changed
gamma = 0.25
tick_values = data_min + (data_max - data_min) * (tick_positions ** (1/gamma))

rounded_labels = [f"{v:.02f}" for v in tick_values]

cbar.set_ticks(tick_values)
cbar.set_ticklabels(rounded_labels)



# plt.pcolormesh(heatmap_data.columns, heatmap_data.index, heatmap_data.values,
#                shading='auto', cmap='inferno')
# plt.colorbar(label='$\sigma$ of PR')
plt.xlabel('Heterogeneity of Synaptic Mean (HSM)', labelpad = 20)
plt.ylabel('Gain (g)')
# plt.title('Uncertainty of PR')
# plt.title('Heatmap of MLEs vs HSMs per Gains')

#for increasing the width of the plot edge line
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(2)

#for increasing the width of the edge line of the colour bar
cbar.outline.set_linewidth(2)

timestamp = datetime.datetime.now().strftime("%Y-%m-%d_ %H;%M;%S")
filename = f"LLE_1000N_0.2s_shortened_uncertainty_{timestamp}.png"
filepath = os.path.join(r"C:\Users\Lenovo\Documents\B. Master Neurophysics\Stage\Presentations and reports and code\code\LLE\MLE map PNGs for Gain and HSM", filename)
plt.savefig(filepath, dpi=300)

plt.show()

# Plotting Trails per Gain

In [ ]:
# FIX: `cbar.outline.set_linewidth(2)` referenced `cbar`, which is never
# created anywhere in this cell (no colorbar here -- this is a line plot,
# not a heatmap). Same leftover-copy-paste bug as in the PR notebook's
# equivalent cell; removed.
plt.rcParams['axes.titlesize'] = 20
plt.rcParams['axes.labelsize'] = 24    # x/y labels
plt.rcParams['xtick.labelsize'] = 20   # x ticks
plt.rcParams['ytick.labelsize'] = 20   # y ticks
plt.rcParams['legend.fontsize'] = 10   # legend text
plt.rcParams['figure.titlesize'] = 16  # suptitle (global title)
plt.rcParams['legend.title_fontsize'] = 16 # legend title

#tick size:
plt.rcParams['xtick.major.size'] = 6    # length of major ticks
plt.rcParams['ytick.major.size'] = 6
plt.rcParams['xtick.major.width'] = 2  # width of major ticks
plt.rcParams['ytick.major.width'] = 2

unique_Gains = np.unique(Gains_sorted)
print(unique_Gains)

unique_HSMs = np.unique(HSMs_sorted)

print(unique_HSMs)

# Colors for weight seeds


weights_seeds = np.unique(Weights_Seeds_sorted)
print(weights_seeds)

color_map = {ws: plt.cm.tab10(i % 10) for i, ws in enumerate(weights_seeds)}

# Linestyles for initialisation seeds
init_seeds_unique = np.unique(Initialisation_Seeds_sorted)
print(init_seeds_unique)

linestyle_tuple = [
    ('solid', 'solid'),
    ('dotted', 'dotted'),
    ('dashed', 'dashed'),
    ('dashdot', 'dashdot'),
    ('loosely dotted', (0, (1, 10))),
    ('densely dotted', (0, (1, 1))),
    ('loosely dashed', (0, (5, 10))),
    ('densely dashed', (0, (5, 1))),
    ('dashdotted', (0, (3, 5, 1, 5))),
    ('densely dashdotted', (0, (3, 1, 1, 1)))
]
linestyle_map = {is_: linestyle_tuple[i % len(linestyle_tuple)][1]
                 for i, is_ in enumerate(init_seeds_unique)} #not completely sure how this works, but I know what it gives qua result



# Loop over all combinations of seeds and gains
for i, gain in enumerate(unique_Gains):
    
    plt.figure(figsize=(12,5))
    for w_seed in weights_seeds:

        if w_seed in weights_seeds:
        
            for i_seed in init_seeds_unique:
                # Find indices matching this combination
                mask = (Weights_Seeds_sorted == w_seed) & (Initialisation_Seeds_sorted == i_seed) & (Gains_sorted == gain)
                if not np.any(mask):
                    continue
        
                # Extract LLEs for this pair
                lles = np.array(LLEs_sorted)[mask]
        
                # Sanity check: must align with unique_HSMs length
                if len(lles) != len(unique_HSMs):
                    print(f"Skipping w={w_seed}, i={i_seed}, wrong length {len(lles)}")
                    continue
        
                # Plot
                
                plt.plot(unique_HSMs[:-1], lles[:-1],
                         color=color_map[w_seed],
                         linestyle=linestyle_map[i_seed],
                         marker='o')
                         #label=f"W={w_seed}, I={i_seed}") put this line back if you wanna see all the init seeds in the legend as well

    #Create legend handles only for weight seeds (colours)
    handles = [plt.Line2D([0], [0], color=color_map[w], lw=3, label=f"W={w}") for w in weights_seeds]

    
    
    plt.xlabel("Heterogeneity of Synaptic Mean (HSM)")
    plt.ylabel("MLE")
    plt.ylim(-130, 130)
    #plt.title(f"Maximal Lyapunov Exponent for 1000 N RNN and gain of {gain:.1f}")
    #plt.grid(True)
    plt.legend(handles=handles, title="Weight Seeds*:", bbox_to_anchor=(1.05, 1))
    plt.tight_layout()

     #for increasing the width of the plot edge line
    ax = plt.gca()
    for spine in ax.spines.values():
        spine.set_linewidth(2)
    
    filename = f"MLE Figure {i} for gain {gain:.1f}.png"
    #change path depending on 10% connectivity or not
    filepath = os.path.join(r"C:\Users\Lenovo\Documents\B. Master Neurophysics\Stage\Presentations and reports and code\code\LLE\MLE across MSH and W and I seeds for different gains", filename)
    plt.subplots_adjust(left = 0.2, right=0.8, top = 0.9, bottom = 0.12) #leaving space for legend
    plt.savefig(filepath, dpi = 300, bbox_inches='tight')
    #plt.savefig(filepath, dpi=300, bbox_inches='tight') #save each figure
    
    plt.show()


# Video Making

In [208]:


# Path where your saved figures are
folder = r"C:\Users\Lenovo\Documents\B. Master Neurophysics\Stage\Presentations and reports and code\code\LLE\MLE across MSH and W and I seeds for different gains"

# Collect all PNGs and sort them nicely
images = sorted(
    [img for img in os.listdir(folder) if img.endswith(".png")],
    key=lambda x: float(x.split("gain")[-1].replace(".png", "").strip())  # optional: sort by gain value
)

# Output video path
output_path = os.path.join(folder, "MLE_movie_1000N.mp4")

# Make the video (fps controls speed)
with imageio.get_writer(output_path,fps=3) as writer:
    for img_name in images:
        img_path = os.path.join(folder, img_name)
        writer.append_data(imageio.imread(img_path))

print("✅ Video saved at:", output_path)


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (3306, 1464) to (3312, 1472) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


✅ Video saved at: C:\Users\Lenovo\Documents\B. Master Neurophysics\Stage\Presentations and reports and code\code\LLE\MLE across MSH and W and I seeds for different gains\MLE_movie_1000N.mp4
